In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModel
import numpy as np
import gc
import warnings
import random
import os
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime
import glob
from tqdm import tqdm

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class DeepSeekCoderForPlagiarismDetection(nn.Module):
    def __init__(self, model_name="deepseek-ai/deepseek-coder-1.3b-base", freeze_layers=8):
        super().__init__()
        
        print(f"Loading {model_name}...")
        self.encoder = AutoModel.from_pretrained(model_name)
        
        hidden_size = self.encoder.config.hidden_size
        
        if freeze_layers > 0 and hasattr(self.encoder, 'layers'):
            num_layers = len(self.encoder.layers)
            print(f"Number of layers: {num_layers}")
            for i in range(min(freeze_layers, num_layers)):
                for param in self.encoder.layers[i].parameters():
                    param.requires_grad = False
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 2)
        )
        
        print(f"Model loaded. Hidden size: {hidden_size}")
        
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.parameters())
        print(f"Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
        print(f"Total parameters: {total_params:,}")
    
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        
        last_hidden_state = outputs.last_hidden_state
        
        attention_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * attention_mask_expanded, dim=1)
        sum_mask = torch.clamp(attention_mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask
        
        logits = self.classifier(mean_pooled)
        
        return logits

class DeepSeekCoderEnhanced(nn.Module):
    def __init__(self, model_name="deepseek-ai/deepseek-coder-1.3b-base", freeze_layers=8):
        super().__init__()
        
        print(f"Loading {model_name}...")
        self.encoder = AutoModel.from_pretrained(model_name)
        
        hidden_size = self.encoder.config.hidden_size
        
        if freeze_layers > 0 and hasattr(self.encoder, 'layers'):
            num_layers = len(self.encoder.layers)
            for i in range(min(freeze_layers, num_layers)):
                for param in self.encoder.layers[i].parameters():
                    param.requires_grad = False
        
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, 2)
        )
        
        print(f"Enhanced model loaded. Hidden size: {hidden_size}")
        
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.parameters())
        print(f"Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
    
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        
        hidden_states = outputs.last_hidden_state
        
        attention_weights = self.attention(hidden_states)
        attention_weights = attention_weights.squeeze(-1)
        
        attention_weights = attention_weights.masked_fill(attention_mask == 0, -1e9)
        attention_weights = torch.softmax(attention_weights, dim=1)
        
        attention_weights = attention_weights.unsqueeze(-1)
        pooled_output = torch.sum(hidden_states * attention_weights, dim=1)
        
        logits = self.classifier(pooled_output)
        
        return logits

class PlagiarismDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.codes = df["clean_code"].fillna("").astype(str).tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.codes)
    
    def __getitem__(self, idx):
        code = self.codes[idx]
        label = self.labels[idx]
        
        inputs = self.tokenizer(
            code,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def train_deepseekcoder(model, train_loader, optimizer, criterion, device, epochs, seed, gradient_accumulation_steps=8):
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        optimizer.zero_grad()
        
        pbar = tqdm(train_loader, desc=f"DeepSeek-Coder - Epoch {epoch+1}/{epochs} (Seed {seed})")
        for step, batch in enumerate(pbar, 1):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss = loss / gradient_accumulation_steps
            loss.backward()
            
            if step % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
            
            total_loss += loss.item() * gradient_accumulation_steps
            
            _, predicted = torch.max(logits, 1)
            batch_total = labels.size(0)
            batch_correct = (predicted == labels).sum().item()
            
            total += batch_total
            correct += batch_correct
            
            avg_loss = total_loss / step
            batch_acc = 100 * batch_correct / batch_total
            pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'acc': f'{batch_acc:.2f}%'})
            
            if step % 20 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        epoch_acc = 100 * correct / total if total > 0 else 0
        avg_loss = total_loss / len(train_loader) if len(train_loader) > 0 else 0
        print(f"[Seed {seed}, DeepSeek-Coder] Epoch {epoch+1} Loss: {avg_loss:.4f}, Acc: {epoch_acc:.2f}%")
    
    return model

def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()
            
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels)
    
    if len(all_labels) > 0:
        accuracy = accuracy_score(all_labels, all_preds)
        precision = precision_score(all_labels, all_preds, average='binary', zero_division=0)
        recall = recall_score(all_labels, all_preds, average='binary', zero_division=0)
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    else:
        accuracy = precision = recall = f1 = 0.0
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': all_preds,
        'labels': all_labels
    }

def evaluate_all_test_files(model, tokenizer, device, seed, model_name="deepseekcoder"):
    results = {}
    
    for i in range(10):
        test_file = f"Test_{i}.csv"
        if os.path.exists(test_file):
            try:
                df = pd.read_csv(test_file, usecols=["clean_code", "label"])
                df["clean_code"] = df["clean_code"].fillna("").astype(str)
                dataset = PlagiarismDataset(df, tokenizer)
                loader = DataLoader(dataset, batch_size=2, shuffle=False)
                
                metrics = evaluate_model(model, loader, device)
                results[test_file] = metrics
                
                print(f"[Seed {seed}] {test_file}:")
                print(f"  Accuracy: {metrics['accuracy']:.4f}")
                print(f"  Precision: {metrics['precision']:.4f}")
                print(f"  Recall: {metrics['recall']:.4f}")
                print(f"  F1-Score: {metrics['f1']:.4f}")
                
            except Exception as e:
                print(f"Error evaluating {test_file}: {e}")
                results[test_file] = None
        else:
            print(f"File {test_file} not found.")
            results[test_file] = None
    
    valid_results = [v for v in results.values() if v is not None]
    if valid_results:
        avg_results = {
            'accuracy': np.mean([r['accuracy'] for r in valid_results]),
            'precision': np.mean([r['precision'] for r in valid_results]),
            'recall': np.mean([r['recall'] for r in valid_results]),
            'f1': np.mean([r['f1'] for r in valid_results]),
        }
        return avg_results, results
    else:
        return None, results

def run_deepseekcoder_experiment_with_seed(seed, train_file, epochs=3, model_size="1.3b", enhanced=False, freeze_layers=8):
    print(f"\n{'='*80}")
    print(f"RUNNING DEEPSEEK-CODER EXPERIMENT WITH SEED: {seed}")
    print(f"{'='*80}")
    
    set_seed(seed)
    
    if model_size == "1.3b":
        model_name = "deepseek-ai/deepseek-coder-1.3b-base"
    elif model_size == "6.7b":
        model_name = "deepseek-ai/deepseek-coder-6.7b-base"
    elif model_size == "33b":
        model_name = "deepseek-ai/deepseek-coder-33b-base"
    else:
        model_name = "deepseek-ai/deepseek-coder-1.3b-base"
    
    print(f"Using model: {model_name}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    train_df = pd.read_csv(train_file, usecols=["clean_code", "label"])
    train_dataset = PlagiarismDataset(train_df, tokenizer)
    
    batch_size = 1
    gradient_accumulation_steps = 16
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True if torch.cuda.is_available() else False,
        num_workers=0
    )
    
    print(f"Batch size: {batch_size}")
    print(f"Gradient accumulation steps: {gradient_accumulation_steps}")
    print(f"Effective batch size: {batch_size * gradient_accumulation_steps}")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if enhanced:
        model = DeepSeekCoderEnhanced(model_name=model_name, freeze_layers=freeze_layers).to(device)
        model_type = "deepseekcoder_enhanced"
    else:
        model = DeepSeekCoderForPlagiarismDetection(model_name=model_name, freeze_layers=freeze_layers).to(device)
        model_type = "deepseekcoder"
    
    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-5,
        weight_decay=0.01
    )
    
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
        eta_min=1e-6
    )
    
    criterion = nn.CrossEntropyLoss()
    
    model = train_deepseekcoder(model, train_loader, optimizer, criterion, device, 
                                epochs, seed, gradient_accumulation_steps)
    
    print(f"\n[Seed {seed}] Evaluating DeepSeek-Coder Model on test files...")
    deepseekcoder_avg_results, deepseekcoder_detailed_results = evaluate_all_test_files(
        model, tokenizer, device, seed, model_type
    )
    
    if deepseekcoder_avg_results:
        print(f"\n[Seed {seed}] DeepSeek-Coder Average Results:")
        print(f"  Accuracy: {deepseekcoder_avg_results['accuracy']:.4f}")
        print(f"  Precision: {deepseekcoder_avg_results['precision']:.4f}")
        print(f"  Recall: {deepseekcoder_avg_results['recall']:.4f}")
        print(f"  F1-Score: {deepseekcoder_avg_results['f1']:.4f}")
    
    os.makedirs(f"models/deepseekcoder_{model_size}", exist_ok=True)
    model_path = f"models/deepseekcoder_{model_size}/deepseekcoder_{'enhanced_' if enhanced else ''}seed{seed}.pt"
    torch.save(model.state_dict(), model_path)
    print(f"[Seed {seed}] DeepSeek-Coder model saved to: {model_path}")
    
    return {
        'seed': seed,
        'model_size': model_size,
        'enhanced': enhanced,
        'deepseekcoder_avg_results': deepseekcoder_avg_results,
        'deepseekcoder_detailed_results': deepseekcoder_detailed_results
    }

def load_codebert_results():
    codebert_results = {}
    
    sources = [
        "results/aggregated_results.csv",
        *glob.glob("results/*final*.csv"),
        *glob.glob("results/*aggregated*.csv"),
        *glob.glob("results/interim_results_*.csv")
    ]
    
    for source in sources:
        if os.path.exists(source) and os.path.isfile(source):
            try:
                df = pd.read_csv(source)
                for _, row in df.iterrows():
                    seed = int(row['seed'])
                    if 'stage2_accuracy' in df.columns:
                        codebert_results[seed] = {
                            'accuracy': row['stage2_accuracy'],
                            'precision': row.get('stage2_precision', row.get('precision', 0)),
                            'recall': row.get('stage2_recall', row.get('recall', 0)),
                            'f1': row.get('stage2_f1', row.get('f1', 0))
                        }
                    elif 'accuracy' in df.columns:
                        codebert_results[seed] = {
                            'accuracy': row['accuracy'],
                            'precision': row.get('precision', 0),
                            'recall': row.get('recall', 0),
                            'f1': row.get('f1', 0)
                        }
                print(f"Loaded CodeBERT results from {os.path.basename(source)}")
                if codebert_results:
                    return codebert_results
            except:
                continue
    
    print("No automatic CodeBERT results found.")
    print("Options:")
    print("1. Enter path to CodeBERT results CSV")
    print("2. Continue without comparison")
    
    choice = input("\nEnter choice (1 or 2): ").strip()
    
    if choice == "1":
        results_path = input("Enter path to CodeBERT results CSV file: ").strip()
        if os.path.exists(results_path):
            try:
                df = pd.read_csv(results_path)
                for _, row in df.iterrows():
                    seed = int(row['seed'])
                    codebert_results[seed] = {
                        'accuracy': row.get('accuracy', row.get('Accuracy', row.get('acc', 0))),
                        'precision': row.get('precision', row.get('Precision', 0)),
                        'recall': row.get('recall', row.get('Recall', 0)),
                        'f1': row.get('f1', row.get('F1', 0))
                    }
                print(f"Loaded CodeBERT results for {len(codebert_results)} seeds")
            except Exception as e:
                print(f"Error loading file: {e}")
        else:
            print("File not found.")
    
    return codebert_results

def perform_comparison_analysis(deepseekcoder_results, codebert_results, metric='accuracy'):
    common_seeds = sorted(set(deepseekcoder_results.keys()) & set(codebert_results.keys()))
    
    if len(common_seeds) < 2:
        print(f"\nNot enough common seeds for comparison (need at least 2, got {len(common_seeds)})")
        print(f"DeepSeek-Coder seeds: {sorted(deepseekcoder_results.keys())}")
        print(f"CodeBERT seeds: {sorted(codebert_results.keys())}")
        return None
    
    deepseekcoder_metrics = [deepseekcoder_results[seed][metric] for seed in common_seeds]
    codebert_metrics = [codebert_results[seed][metric] for seed in common_seeds]
    
    print(f"\n{'='*80}")
    print(f"COMPARISON ANALYSIS - {metric.upper()} METRIC")
    print(f"{'='*80}")
    print(f"Comparing {len(common_seeds)} common seeds: {common_seeds}")
    
    print(f"\nDeepSeek-Coder {metric} across {len(common_seeds)} seeds:")
    print(f"  Mean: {np.mean(deepseekcoder_metrics):.4f}")
    print(f"  Std: {np.std(deepseekcoder_metrics):.4f}")
    print(f"  Min: {np.min(deepseekcoder_metrics):.4f}")
    print(f"  Max: {np.max(deepseekcoder_metrics):.4f}")
    
    print(f"\nCodeBERT Stage 2 {metric} across {len(common_seeds)} seeds:")
    print(f"  Mean: {np.mean(codebert_metrics):.4f}")
    print(f"  Std: {np.std(codebert_metrics):.4f}")
    print(f"  Min: {np.min(codebert_metrics):.4f}")
    print(f"  Max: {np.max(codebert_metrics):.4f}")
    
    differences = np.array(deepseekcoder_metrics) - np.array(codebert_metrics)
    print(f"\nDifferences (DeepSeek-Coder - CodeBERT) in {metric}:")
    print(f"  Mean difference: {np.mean(differences):.4f}")
    print(f"  Std of difference: {np.std(differences):.4f}")
    print(f"  Min difference: {np.min(differences):.4f}")
    print(f"  Max difference: {np.max(differences):.4f}")
    print(f"  DeepSeek-Coder better: {sum(i > 0 for i in differences)}/{len(differences)} seeds")
    print(f"  CodeBERT better: {sum(i < 0 for i in differences)}/{len(differences)} seeds")
    print(f"  Equal: {sum(i == 0 for i in differences)}/{len(differences)} seeds")
    
    t_stat, p_value = stats.ttest_rel(deepseekcoder_metrics, codebert_metrics)
    print(f"\nPaired t-test results (DeepSeek-Coder vs CodeBERT):")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {p_value:.6f}")
    
    try:
        w_stat, w_p_value = stats.wilcoxon(deepseekcoder_metrics, codebert_metrics)
        print(f"\nWilcoxon signed-rank test results:")
        print(f"  W-statistic: {w_stat:.4f}")
        print(f"  p-value: {w_p_value:.6f}")
    except:
        print(f"\nWilcoxon test could not be performed")
        w_stat, w_p_value = 0, 1.0
    
    if differences.std() > 0:
        d = np.mean(differences) / np.std(differences)
    else:
        d = 0
    print(f"\nEffect size (Cohen's d): {d:.4f}")
    
    print(f"\nStatistical Significance Interpretation:")
    if p_value < 0.05:
        if np.mean(differences) > 0:
            print(f"  ✅ DeepSeek-Coder is SIGNIFICANTLY BETTER than CodeBERT (p < 0.05)")
        else:
            print(f"  ✅ CodeBERT is SIGNIFICANTLY BETTER than DeepSeek-Coder (p < 0.05)")
        
        if abs(d) >= 0.8:
            print(f"  ✅ Large effect size (|d| = {abs(d):.2f})")
        elif abs(d) >= 0.5:
            print(f"  ⚠️  Medium effect size (|d| = {abs(d):.2f})")
        else:
            print(f"  ⚠️  Small effect size (|d| = {abs(d):.2f})")
    else:
        print(f"  ❌ No statistically significant difference (p = {p_value:.4f})")
    
    return {
        'metric': metric,
        'common_seeds': common_seeds,
        'deepseekcoder_mean': float(np.mean(deepseekcoder_metrics)),
        'deepseekcoder_std': float(np.std(deepseekcoder_metrics)),
        'codebert_mean': float(np.mean(codebert_metrics)),
        'codebert_std': float(np.std(codebert_metrics)),
        'difference_mean': float(np.mean(differences)),
        'difference_std': float(np.std(differences)),
        't_statistic': float(t_stat),
        'p_value': float(p_value),
        'w_statistic': float(w_stat),
        'wilcoxon_p': float(w_p_value),
        'cohens_d': float(d),
        'deepseekcoder_better': int(sum(i > 0 for i in differences)),
        'codebert_better': int(sum(i < 0 for i in differences)),
        'equal': int(sum(i == 0 for i in differences))
    }

def create_comparison_visualizations(deepseekcoder_metrics, codebert_metrics, differences, seeds_used, save_dir, metric_name="Accuracy", model_name="DeepSeek-Coder"):
    seeds = seeds_used
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    axes[0, 0].plot(seeds, deepseekcoder_metrics, 'o-', label=model_name, linewidth=2, markersize=8, color='purple')
    axes[0, 0].plot(seeds, codebert_metrics, 's-', label='CodeBERT Stage 2', linewidth=2, markersize=8, color='green')
    axes[0, 0].set_xlabel('Seed', fontsize=12)
    axes[0, 0].set_ylabel(metric_name, fontsize=12)
    axes[0, 0].set_title(f'{metric_name} Comparison Across Seeds', fontsize=14, fontweight='bold')
    axes[0, 0].legend(fontsize=12, loc='best')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_xticks(seeds)
    
    data_to_plot = [deepseekcoder_metrics, codebert_metrics]
    bp = axes[0, 1].boxplot(data_to_plot, labels=[model_name, 'CodeBERT\nStage 2'], patch_artist=True)
    
    colors = ['lavender', 'lightgreen']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    axes[0, 1].set_ylabel(metric_name, fontsize=12)
    axes[0, 1].set_title(f'Distribution Comparison', fontsize=14, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    for i, data in enumerate(data_to_plot):
        axes[0, 1].plot(i+1, np.mean(data), 'r_', markersize=15, markeredgewidth=2)
        axes[0, 1].text(i+1, np.mean(data) + 0.01, f'{np.mean(data):.3f}', 
                       ha='center', va='bottom', fontweight='bold')
    
    bars = axes[1, 0].bar(seeds, differences, 
                         color=['purple' if x >= 0 else 'green' for x in differences], alpha=0.7)
    axes[1, 0].axhline(y=0, color='black', linestyle='-', alpha=0.5)
    axes[1, 0].set_xlabel('Seed', fontsize=12)
    axes[1, 0].set_ylabel(f'Difference ({model_name} - CodeBERT)', fontsize=12)
    axes[1, 0].set_title(f'Performance Difference Across Seeds', fontsize=14, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    axes[1, 0].set_xticks(seeds)
    
    for bar, diff in zip(bars, differences):
        height = bar.get_height()
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + (0.01 if height >= 0 else -0.02),
                       f'{diff:.4f}', ha='center', va='bottom' if height >= 0 else 'top', fontsize=9)
    
    avg_difference = np.mean(differences)
    axes[1, 0].axhline(y=avg_difference, color='blue', linestyle='--', 
                      linewidth=2, label=f'Average: {avg_difference:.4f}')
    axes[1, 0].legend()
    
    axes[1, 1].hist(differences, bins=min(10, len(differences)), edgecolor='black', alpha=0.7, color='skyblue')
    axes[1, 1].axvline(x=0, color='black', linestyle='-', linewidth=2, alpha=0.7)
    axes[1, 1].axvline(x=avg_difference, color='blue', linestyle='--', 
                      linewidth=2, label=f'Mean: {avg_difference:.4f}')
    axes[1, 1].set_xlabel(f'Difference ({model_name} - CodeBERT)', fontsize=12)
    axes[1, 1].set_ylabel('Frequency', fontsize=12)
    axes[1, 1].set_title(f'Distribution of Differences', fontsize=14, fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    fig.text(0.02, 0.02, 
             f'Summary: {model_name} Mean = {np.mean(deepseekcoder_metrics):.4f} ± {np.std(deepseekcoder_metrics):.4f}, '
             f'CodeBERT Mean = {np.mean(codebert_metrics):.4f} ± {np.std(codebert_metrics):.4f}, '
             f'Avg Difference = {avg_difference:.4f}',
             fontsize=10, style='italic', bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray", alpha=0.5))
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.97])
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    fig_path_png = os.path.join(save_dir, f'comparison_{metric_name.lower()}_{timestamp}.png')
    fig_path_pdf = os.path.join(save_dir, f'comparison_{metric_name.lower()}_{timestamp}.pdf')
    
    plt.savefig(fig_path_png, dpi=300, bbox_inches='tight')
    plt.savefig(fig_path_pdf, bbox_inches='tight')
    plt.close()
    
    print(f"\nComparison visualizations saved to:")
    print(f"  {fig_path_png}")
    print(f"  {fig_path_pdf}")
    
    return fig_path_png

def main():
    print("="*80)
    print("DEEPSEEK-CODER VS CODEBERT - PLAGIARISM DETECTION COMPARISON")
    print("="*80)
    
    train_file = "Train.csv"
    
    if not os.path.exists(train_file):
        print(f"Error: Train file '{train_file}' not found!")
        print("Please ensure Train.csv exists in the current directory.")
        return
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        torch.cuda.empty_cache()
    
    print("\nLoading CodeBERT Stage 2 results...")
    codebert_results = load_codebert_results()
    
    if not codebert_results:
        print("Warning: No CodeBERT results found. Will only run DeepSeek-Coder experiments.")
        compare_with_codebert = False
    else:
        compare_with_codebert = True
        print(f"Loaded CodeBERT results for seeds: {sorted(codebert_results.keys())}")
    
    print("\n" + "="*80)
    print("DEEPSEEK-CODER MODEL SELECTION")
    print("="*80)
    print("Available DeepSeek-Coder models:")
    print("1. deepseek-coder-1.3b-base (1.3B parameters, requires ~3GB GPU memory)")
    print("2. deepseek-coder-6.7b-base (6.7B parameters, requires ~14GB GPU memory)")
    print("3. deepseek-coder-33b-base (33B parameters, requires ~66GB GPU memory)")
    
    model_choice = input("\nSelect model size (1, 2, or 3): ").strip()
    if model_choice == "1":
        model_size = "1.3b"
    elif model_choice == "2":
        model_size = "6.7b"
    elif model_choice == "3":
        model_size = "33b"
    else:
        print("Invalid choice. Using deepseek-coder-1.3b-base (default).")
        model_size = "1.3b"
    
    print("\n" + "="*80)
    print("ARCHITECTURE SELECTION")
    print("="*80)
    print("Available architectures:")
    print("1. Basic DeepSeek-Coder (mean pooling)")
    print("2. Enhanced DeepSeek-Coder (attention pooling)")
    
    arch_choice = input("\nSelect architecture (1 or 2): ").strip()
    enhanced = (arch_choice == "2")
    
    print("\n" + "="*80)
    print("LAYER FREEZING")
    print("="*80)
    print("To save GPU memory, we can freeze some layers.")
    try:
        freeze_layers = int(input("Number of layers to freeze (0-20, recommended 8-12): ") or "8")
        freeze_layers = max(0, min(freeze_layers, 20))
    except:
        freeze_layers = 8
    
    DEFAULT_SEEDS = [42, 123, 456, 789, 999, 111, 222, 333, 444, 555]
    
    if compare_with_codebert:
        codebert_seeds = sorted(codebert_results.keys())
        print(f"\nCodeBERT was run on {len(codebert_seeds)} seeds: {codebert_seeds}")
        print(f"Default seeds available: {DEFAULT_SEEDS}")
        
        use_same_seeds = input(f"\nRun DeepSeek-Coder on same seeds as CodeBERT? (y/n): ").strip().lower()
        
        if use_same_seeds == 'y':
            seeds_to_run = codebert_seeds
            print(f"Will run DeepSeek-Coder on seeds: {seeds_to_run}")
        else:
            print(f"\nAvailable default seeds: {DEFAULT_SEEDS}")
            seeds_input = input(f"Enter seeds to run (comma-separated, e.g., '42,123'): ").strip()
            if seeds_input:
                try:
                    seeds_to_run = [int(s.strip()) for s in seeds_input.split(',')]
                except:
                    print("Invalid input. Using first 2 default seeds.")
                    seeds_to_run = DEFAULT_SEEDS[:2]
            else:
                seeds_to_run = DEFAULT_SEEDS[:2]
    else:
        print(f"\nDefault seeds available: {DEFAULT_SEEDS}")
        seeds_input = input(f"Enter seeds to run (comma-separated, press Enter for first 2): ").strip()
        if seeds_input:
            try:
                seeds_to_run = [int(s.strip()) for s in seeds_input.split(',')]
            except:
                print("Invalid input. Using first 2 default seeds.")
                seeds_to_run = DEFAULT_SEEDS[:2]
        else:
            seeds_to_run = DEFAULT_SEEDS[:2]
    
    try:
        epochs = int(input("\nNumber of training epochs for DeepSeek-Coder (recommended: 1-3): ") or "2")
        epochs = max(1, min(epochs, 5))
    except:
        epochs = 2
    
    print(f"\n{'='*80}")
    print("EXPERIMENT CONFIGURATION")
    print("="*80)
    print(f"  Model: DeepSeek-Coder {model_size}")
    print(f"  Architecture: {'Enhanced (attention pooling)' if enhanced else 'Basic (mean pooling)'}")
    print(f"  Frozen layers: {freeze_layers}")
    print(f"  Seeds: {seeds_to_run}")
    print(f"  Epochs: {epochs}")
    print(f"  Compare with CodeBERT: {compare_with_codebert}")
    print("\n⚠️  WARNING: DeepSeek-Coder requires significant GPU memory!")
    print(f"   Model size: {model_size}")
    print(f"   Estimated GPU memory needed: {'~3GB' if model_size == '1.3b' else '~14GB' if model_size == '6.7b' else '~66GB'}")
    
    confirm = input("\nProceed with DeepSeek-Coder experiments? (y/n): ").strip().lower()
    if confirm != 'y':
        print("Experiment cancelled.")
        return
    
    model_dir = f"models/deepseekcoder_{model_size}{'_enhanced' if enhanced else ''}"
    results_dir = f"results/deepseekcoder_{model_size}{'_enhanced' if enhanced else ''}"
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)
    
    print(f"\n{'='*80}")
    print(f"STARTING DEEPSEEK-CODER EXPERIMENTS")
    print(f"{'='*80}")
    
    all_deepseekcoder_results = {}
    deepseekcoder_accuracies = []
    deepseekcoder_f1_scores = []
    
    start_time = datetime.now()
    
    for i, seed in enumerate(seeds_to_run, 1):
        print(f"\n[{i}/{len(seeds_to_run)}] ", end="")
        try:
            result = run_deepseekcoder_experiment_with_seed(
                seed, train_file, epochs, model_size, enhanced, freeze_layers
            )
            
            if result['deepseekcoder_avg_results']:
                all_deepseekcoder_results[seed] = result['deepseekcoder_avg_results']
                deepseekcoder_accuracies.append(result['deepseekcoder_avg_results']['accuracy'])
                deepseekcoder_f1_scores.append(result['deepseekcoder_avg_results']['f1'])
            
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            interim_results = []
            for s, res in all_deepseekcoder_results.items():
                interim_results.append({
                    'seed': s,
                    'model_size': model_size,
                    'enhanced': enhanced,
                    'frozen_layers': freeze_layers,
                    'accuracy': res['accuracy'],
                    'precision': res['precision'],
                    'recall': res['recall'],
                    'f1': res['f1']
                })
            
            if interim_results:
                interim_df = pd.DataFrame(interim_results)
                interim_path = f"{results_dir}/deepseekcoder_interim_results_{timestamp}.csv"
                interim_df.to_csv(interim_path, index=False)
                print(f"[Progress] DeepSeek-Coder interim results saved to: {interim_path}")
                
        except torch.cuda.OutOfMemoryError:
            print(f"\n❌ OUT OF MEMORY ERROR for seed {seed}")
            print(f"   DeepSeek-Coder {model_size} requires too much GPU memory.")
            print(f"   Try using a smaller model or reducing batch size.")
            break
        except Exception as e:
            print(f"\n❌ ERROR for seed {seed}: {e}")
            continue
    
    end_time = datetime.now()
    total_time = (end_time - start_time).total_seconds() / 60
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if all_deepseekcoder_results:
        final_results = []
        for seed, res in all_deepseekcoder_results.items():
            final_results.append({
                'seed': seed,
                'model_size': model_size,
                'enhanced': enhanced,
                'frozen_layers': freeze_layers,
                'accuracy': res['accuracy'],
                'precision': res['precision'],
                'recall': res['recall'],
                'f1': res['f1']
            })
        
        results_df = pd.DataFrame(final_results)
        results_path = f"{results_dir}/deepseekcoder_final_results_{timestamp}.csv"
        results_df.to_csv(results_path, index=False)
        
        print(f"\n{'='*80}")
        print("DEEPSEEK-CODER RESULTS SUMMARY")
        print(f"{'='*80}")
        print(f"Experiments completed in {total_time:.2f} minutes")
        print(f"Number of seeds completed: {len(all_deepseekcoder_results)}")
        print(f"Model: DeepSeek-Coder {model_size}")
        print(f"Architecture: {'Enhanced (attention pooling)' if enhanced else 'Basic (mean pooling)'}")
        print(f"Frozen layers: {freeze_layers}")
        
        if deepseekcoder_accuracies:
            print(f"\nDeepSeek-Coder Average Accuracy:")
            print(f"  Mean: {np.mean(deepseekcoder_accuracies):.4f} ± {np.std(deepseekcoder_accuracies):.4f}")
            print(f"  Range: [{np.min(deepseekcoder_accuracies):.4f}, {np.max(deepseekcoder_accuracies):.4f}]")
        
        if deepseekcoder_f1_scores:
            print(f"\nDeepSeek-Coder Average F1-Score:")
            print(f"  Mean: {np.mean(deepseekcoder_f1_scores):.4f} ± {np.std(deepseekcoder_f1_scores):.4f}")
            print(f"  Range: [{np.min(deepseekcoder_f1_scores):.4f}, {np.max(deepseekcoder_f1_scores):.4f}]")
        
        print(f"\n📊 DeepSeek-Coder results saved to: {results_path}")
        
        print(f"\n{'='*80}")
        print("DEEPSEEK-CODER PER-SEED RESULTS")
        print(f"{'='*80}")
        print(f"{'Seed':<8} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
        print(f"{'-'*60}")
        
        for seed in sorted(all_deepseekcoder_results.keys()):
            res = all_deepseekcoder_results[seed]
            print(f"{seed:<8} {res['accuracy']:<12.4f} {res['precision']:<12.4f} "
                  f"{res['recall']:<12.4f} {res['f1']:<12.4f}")
    
    if compare_with_codebert and all_deepseekcoder_results:
        print(f"\n{'='*80}")
        print("COMPARISON WITH CODEBERT STAGE 2")
        print(f"{'='*80}")
        
        comparison_acc = perform_comparison_analysis(all_deepseekcoder_results, codebert_results, metric='accuracy')
        
        comparison_f1 = perform_comparison_analysis(all_deepseekcoder_results, codebert_results, metric='f1')
        
        if comparison_acc and comparison_f1:
            common_seeds = comparison_acc['common_seeds']
            
            deepseekcoder_acc_common = [all_deepseekcoder_results[seed]['accuracy'] for seed in common_seeds]
            codebert_acc_common = [codebert_results[seed]['accuracy'] for seed in common_seeds]
            differences_acc = np.array(deepseekcoder_acc_common) - np.array(codebert_acc_common)
            
            deepseekcoder_f1_common = [all_deepseekcoder_results[seed]['f1'] for seed in common_seeds]
            codebert_f1_common = [codebert_results[seed]['f1'] for seed in common_seeds]
            differences_f1 = np.array(deepseekcoder_f1_common) - np.array(codebert_f1_common)
            
            model_name = f"DeepSeek-Coder {model_size}{' Enhanced' if enhanced else ''}"
            
            fig_path_acc = create_comparison_visualizations(
                deepseekcoder_acc_common, codebert_acc_common, differences_acc, 
                common_seeds, results_dir, metric_name="Accuracy", model_name=model_name
            )
            
            fig_path_f1 = create_comparison_visualizations(
                deepseekcoder_f1_common, codebert_f1_common, differences_f1,
                common_seeds, results_dir, metric_name="F1-Score", model_name=model_name
            )
            
            comparison_stats = {
                'comparison_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'model_name': model_name,
                'common_seeds': common_seeds,
                'accuracy_comparison': comparison_acc,
                'f1_comparison': comparison_f1,
                'summary': {
                    'deepseekcoder_accuracy_mean': float(np.mean(deepseekcoder_acc_common)),
                    'deepseekcoder_accuracy_std': float(np.std(deepseekcoder_acc_common)),
                    'codebert_accuracy_mean': float(np.mean(codebert_acc_common)),
                    'codebert_accuracy_std': float(np.std(codebert_acc_common)),
                    'deepseekcoder_f1_mean': float(np.mean(deepseekcoder_f1_common)),
                    'deepseekcoder_f1_std': float(np.std(deepseekcoder_f1_common)),
                    'codebert_f1_mean': float(np.mean(codebert_f1_common)),
                    'codebert_f1_std': float(np.std(codebert_f1_common))
                }
            }
            
            stats_path = f"{results_dir}/comparison_stats_{timestamp}.json"
            with open(stats_path, 'w') as f:
                json.dump(comparison_stats, f, indent=4)
            
            print(f"\n📈 Comparison statistics saved to: {stats_path}")
            print(f"📸 Comparison visualizations saved in '{results_dir}' directory")
            
            print(f"\n{'='*80}")
            print("COMPARISON SUMMARY TABLE")
            print(f"{'='*80}")
            print(f"{'Seed':<8} {'DeepSeek-Coder Acc':<18} {'CodeBERT Acc':<12} {'Δ Acc':<10} {'DeepSeek-Coder F1':<18} {'CodeBERT F1':<12} {'Δ F1':<10}")
            print(f"{'-'*90}")
            
            for seed in common_seeds:
                deepseekcoder_acc = all_deepseekcoder_results[seed]['accuracy']
                codebert_acc = codebert_results[seed]['accuracy']
                diff_acc = deepseekcoder_acc - codebert_acc
                
                deepseekcoder_f1 = all_deepseekcoder_results[seed]['f1']
                codebert_f1 = codebert_results[seed]['f1']
                diff_f1 = deepseekcoder_f1 - codebert_f1
                
                print(f"{seed:<8} {deepseekcoder_acc:<18.4f} {codebert_acc:<12.4f} {diff_acc:<10.4f} "
                      f"{deepseekcoder_f1:<18.4f} {codebert_f1:<12.4f} {diff_f1:<10.4f}")
    
    print(f"\n{'='*80}")
    print("EXPERIMENT COMPLETE")
    print(f"{'='*80}")
    print(f"Total time: {total_time:.2f} minutes")
    if all_deepseekcoder_results:
        print(f"DeepSeek-Coder models saved in: {model_dir}/")
        print(f"DeepSeek-Coder results saved in: {results_dir}/")
    
    if compare_with_codebert and 'comparison_acc' in locals() and comparison_acc:
        print(f"\nComparison conclusion:")
        if comparison_acc['p_value'] < 0.05:
            if comparison_acc['difference_mean'] > 0:
                print(f"  ✅ DeepSeek-Coder is SIGNIFICANTLY BETTER than CodeBERT for Accuracy")
            else:
                print(f"  ✅ CodeBERT is SIGNIFICANTLY BETTER than DeepSeek-Coder for Accuracy")
        else:
            print(f"  ⚠️  No significant difference between DeepSeek-Coder and CodeBERT for Accuracy")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print("\nGPU memory cleared.")

if __name__ == "__main__":
    main()